# Applying DN-MI to Your Data

This notebook demonstrates how to apply dimensionality normalization (CDF transform) to your own contingency table data for cross-K comparability.

In [ ]:
import numpy as np
from dn_mi import plugin_mi_3d, mi_to_z_cdf, correct_basharin, mi_to_z

## Example 1: Single 3D Contingency Table

Suppose you have a 3D contingency table from a stratified analysis:
- X: genotype (3 categories)
- Y: disease status (2 categories)  
- Z: population strata (10 groups)

In [ ]:
# Your data: shape (k_x, k_y, k_z)
# Replace this with your actual contingency table
table = np.array([
    # Stratum 1
    [[45, 55], [30, 20], [25, 25]],  # Genotypes AA, Aa, aa
    # Stratum 2  
    [[40, 60], [35, 25], [20, 30]],
    # ... (add more strata)
]).transpose(1, 2, 0)  # Shape: (k_x, k_y, k_z)

print(f"Table shape: {table.shape}")
print(f"Total N: {table.sum():.0f}")

In [ ]:
# Calculate plugin MI
mi_raw = plugin_mi_3d(table)
print(f"Raw plugin MI: {mi_raw:.6f} bits")

# Get dimensions
k_x, k_y, k_z = table.shape
N = int(table.sum())

# Apply different corrections
mi_basharin = correct_basharin(mi_raw, k_x, k_y, k_z, N)
z_basic = mi_to_z(mi_raw, k_x, k_y, k_z, N)
z_cdf = mi_to_z_cdf(mi_raw, k_x, k_y, k_z, N)

print(f"\nBasharin corrected MI: {mi_basharin:.6f} bits")
print(f"DN-basic z-score: {z_basic:.3f}")
print(f"CDF transform z-score: {z_cdf:.3f} (recommended)")

## Example 2: Comparing MI Across Different K

The key benefit of the CDF transform is cross-K comparability. Let's compare MI values across different stratification levels.

In [ ]:
from dn_mi import generate_null, build_3d_table

# Generate null data at different k_z values
k_z_values = [5, 10, 20, 50]
N = 10000
k_x, k_y = 3, 2
rng = np.random.default_rng(42)

results = []
for k_z in k_z_values:
    # Generate null data
    g, d, p = generate_null(N, k_x, k_z, rng=rng)
    table = build_3d_table(g, d, p, k_x, k_y, k_z)
    
    # Calculate MI and transforms
    mi_raw = plugin_mi_3d(table)
    z_cdf = mi_to_z_cdf(mi_raw, k_x, k_y, k_z, N)
    
    results.append({
        'k_z': k_z,
        'mi_raw': mi_raw,
        'z_cdf': z_cdf
    })
    
    print(f"k_z={k_z:3d}: MI={mi_raw:.6f} bits, z_CDF={z_cdf:+.3f}")

Notice that:
- Raw MI increases with k_z (dimensionality bias)
- z_CDF values are all ~N(0,1) under the null, regardless of k_z

This enables direct comparison across different stratification levels!

## Example 3: Hypothesis Testing

Use the CDF transform for testing independence.

In [ ]:
from scipy import stats as scipy_stats

# Your data
rng = np.random.default_rng(42)
table = rng.integers(10, 100, size=(3, 2, 10))
k_x, k_y, k_z = table.shape
N = int(table.sum())

# Calculate z-score
mi = plugin_mi_3d(table)
z = mi_to_z_cdf(mi, k_x, k_y, k_z, N)

# Upper-tail p-value for association
p_value = scipy_stats.norm.sf(z)

print(f"MI: {mi:.6f} bits")
print(f"z-score: {z:.3f}")
print(f"p-value: {p_value:.4e}")
print(f"Significant at α=0.05: {p_value < 0.05}")

## Example 4: Working with 2D Tables

For unconditional MI (no stratification), use k_z=1.

In [ ]:
from dn_mi import plugin_mi_2d

# 2D contingency table
table_2d = np.array([
    [120, 80],
    [90, 110],
    [70, 130]
])

k_x, k_y = table_2d.shape
N = int(table_2d.sum())

# Calculate MI
mi = plugin_mi_2d(table_2d)
print(f"Raw MI: {mi:.6f} bits")

# Transform with k_z=1 for unconditional MI
z_cdf = mi_to_z_cdf(mi, k_x, k_y, k_z=1, N=N)
print(f"CDF z-score: {z_cdf:.3f}")

## Summary

**Key takeaways:**

1. Use `plugin_mi_3d()` for conditional MI (stratified data)
2. Use `plugin_mi_2d()` for unconditional MI (2D tables)
3. Use `mi_to_z_cdf()` for cross-K comparability (recommended)
4. The CDF transform gives z ~ N(0,1) under null, enabling standard hypothesis testing
5. Z-scores are directly comparable across different k_x, k_y, k_z values

For more details, see the paper and documentation.